# p53 Mutant Stability Analysis - Explicit Solvent MD (Multi-GPU, Optimized)

This notebook compares the stability of:
- **Wild-type (WT)** p53 core domain (baseline)
- **R175H** cancer mutation (destabilized)
- **R175H + N239Y** rescue mutation combination

## Method
- **Explicit solvent (TIP3P)** - More accurate than implicit solvent
- **4 fs timestep with HMR** - Hydrogen Mass Repartitioning for 2x speedup
- **Smaller water box (0.8 nm)** - ~30% fewer atoms, faster simulation
- **NPT ensemble** - Constant pressure and temperature
- **ESMFold API** - Structure prediction without local installation
- **Multi-GPU parallel** - Runs variants simultaneously on available GPUs

## Speed Optimizations (~2.5x faster)
- HMR allows 4 fs timestep (vs standard 2 fs)
- 0.8 nm water padding (vs standard 1.0 nm)
- Expected: ~70-90 ns/day on T4 GPU

## Installation (Kaggle)
1. Run **cell 1** to install Miniforge + OpenMM with CUDA
2. Run **cell 2** to import and verify CUDA
3. Run remaining cells

In [ ]:
# @title 1. Install Miniforge + OpenMM with CUDA (Run this FIRST)
import os
import subprocess

# Check if we already have our custom conda
if os.path.exists('/kaggle/working/miniforge'):
    print("Miniforge already installed, skipping download...")
else:
    print("Step 1: Downloading Miniforge...")
    !wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O /tmp/miniforge.sh
    
    print("Step 2: Installing Miniforge...")
    !bash /tmp/miniforge.sh -b -p /kaggle/working/miniforge -f

print("Step 3: Setting up PATH...")
os.environ['PATH'] = '/kaggle/working/miniforge/bin:' + os.environ['PATH']

print("Step 4: Installing OpenMM with CUDA...")
!mamba install -y -c conda-forge openmm cudatoolkit=11.8 pdbfixer mdtraj -q

print("Step 5: Installing pip packages...")
!/kaggle/working/miniforge/bin/pip install requests matplotlib

print("\n" + "="*50)
print("INSTALLATION COMPLETE!")
print("="*50)
print("\nNow run cell 2 to import and verify CUDA")

In [ ]:
# @title 2. Import Libraries and Verify CUDA
import sys
sys.path.insert(0, '/kaggle/working/miniforge/lib/python3.12/site-packages')

import requests
import time
import os
import subprocess
import numpy as np
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

from pdbfixer import PDBFixer
from openmm import *
from openmm.app import *
from openmm.unit import *
import mdtraj as md

print("✓ All imports successful!")

# Detect GPU platform
platforms = [Platform.getPlatform(i).getName() for i in range(Platform.getNumPlatforms())]
print(f"\nOpenMM platforms: {platforms}")

# Detect GPUs
try:
    result = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True)
    gpu_lines = [l for l in result.stdout.split('\n') if l.startswith('GPU')]
    N_GPUS = len(gpu_lines)
    print(f"\nDetected {N_GPUS} GPU(s):")
    for g in gpu_lines:
        print(f"  {g}")
except:
    N_GPUS = 0

if 'CUDA' in platforms:
    PLATFORM_NAME = 'CUDA'
    print(f"\n✓ CUDA platform available!")
elif 'OpenCL' in platforms:
    PLATFORM_NAME = 'OpenCL'
    print(f"\n⚠ OpenCL only (CUDA not available)")
else:
    PLATFORM_NAME = 'CPU'
    print(f"\n✗ No GPU platform, using CPU")

print(f"\n>>> Will use: {PLATFORM_NAME} with {N_GPUS} GPU(s) <<<")

In [ ]:
# @title 3. Configuration (10 ns - Top Rescue Mutations)

# === p53 Sequence ===
P53_FULL = (
    "MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLSPDDIEQWFTEDPGP"
    "DEAPRMPEAAPPVAPAPAAPTPAAPAPAPSWPLSSSVPSQKTYQGSYGFRLGFLHSGTAK"
    "SVTCTYSPALNKMFCQLAKTCPVQLWVDSTPPPGTRVRAMAIYKQSQHMTEVVRRCPHHE"
    "RCSDSDGLAPPQHLIRVEGNLRVEYLDDRNTFRHSVVVPYEPPEVGSDCTTIHYNYMCNS"
    "SCMGGMNRRPILTIITLEDSSGNLLGRNSFEVRVCACPGRDRRTEEENLRKKGEPHHELP"
    "PGSTKRALPNNTSSSPQPKKKPLDGEYFTLQIRGRERFEMFRELNEALELKDAQAGKEPG"
    "GSRAHSSHLKSKKGQSTSRHKKLMFKTEGPDSD"
)
P53_CORE = P53_FULL[93:312]  # Core domain (residues 94-312)
CORE_START = 94

# === OPTIMIZED Simulation Parameters ===
TIMESTEP = 4.0                  # fs (4 fs with HMR)
USE_HMR = True                  # Hydrogen Mass Repartitioning
WATER_PADDING = 0.8             # nm
EQUILIBRATION_STEPS = 25000     # 100 ps NPT equilibration
PRODUCTION_STEPS = 2500000      # 10 ns production
SAVE_INTERVAL = 2500            # Save every 10 ps

# === Variants to Simulate ===
# NOTE: Using beam search rescues (all within core domain 94-312)
# FMR best candidates have mutations outside core (G325, D48, etc.)
VARIANTS = {
    # Controls
    'WT': [],                                    # Wild-type baseline
    'R175H': ['R175H'],                          # Cancer mutation (+9.99 kcal/mol)
    
    # Top Beam Search Rescues (within core domain)
    'R175H_S95A': ['R175H', 'S95A'],             # #1: ΔΔG -6.23 kcal/mol
    'R175H_M133L': ['R175H', 'M133L'],           # #2: ΔΔG -5.60 kcal/mol
    'R175H_C229A': ['R175H', 'C229A'],           # #4: ΔΔG -4.40 kcal/mol
}

# === ESMFold API ===
ESMFOLD_API_URL = "https://api.esmatlas.com/foldSequence/v1/pdb/"

# Create output directories
os.makedirs("structures", exist_ok=True)
os.makedirs("trajectories", exist_ok=True)

print("="*60)
print("p53CAD: RESCUE MUTATION MD VALIDATION")
print("="*60)
print(f"\nCore domain: residues {CORE_START}-312 ({len(P53_CORE)} aa)")
print(f"\nVariants to simulate:")

predictions = {'WT': 'baseline', 'R175H': '+9.99', 
               'R175H_S95A': '-6.23', 'R175H_M133L': '-5.60', 'R175H_C229A': '-4.40'}
for name, muts in VARIANTS.items():
    ddg = predictions.get(name, 'N/A')
    print(f"  {name}: {muts if muts else 'WT'} (ΔΔG: {ddg})")

print(f"\nSimulation: {PRODUCTION_STEPS * TIMESTEP / 1e6:.0f} ns explicit solvent")
print(f"Optimizations: HMR (4fs), smaller water box (0.8nm)")
print(f"\nEstimated time: ~2-3 hours on 2x T4 GPUs")

In [ ]:
# @title 4. Helper Functions

def apply_mutations(sequence, mutations):
    """Apply mutations to the p53 core domain sequence."""
    for mut in mutations:
        wt_aa = mut[0]
        pos = int(mut[1:-1])
        mut_aa = mut[-1]
        core_pos = pos - CORE_START

        if sequence[core_pos] != wt_aa:
            raise ValueError(f"Expected {wt_aa} at position {pos}, found {sequence[core_pos]}")

        sequence = sequence[:core_pos] + mut_aa + sequence[core_pos+1:]
        print(f"  Applied {mut} at core position {core_pos}")
    return sequence


def predict_structure_esmfold(sequence, max_retries=3):
    """Predict structure using ESMFold API with retry logic."""
    for attempt in range(max_retries):
        try:
            print(f"  Calling ESMFold API (attempt {attempt + 1}/{max_retries})...")
            response = requests.post(
                ESMFOLD_API_URL,
                data=sequence,
                headers={'Content-Type': 'text/plain'},
                timeout=300
            )
            if response.status_code == 200:
                print("  Success!")
                return response.text
            elif response.status_code == 503:
                print(f"  Server busy, waiting 30s...")
                time.sleep(30)
            else:
                print(f"  Error: {response.status_code}")
                time.sleep(10)
        except requests.Timeout:
            print(f"  Timeout, retrying...")
            time.sleep(10)
    raise RuntimeError("ESMFold API failed after all retries")


print("Helper functions defined.")

In [ ]:
# @title 5. Explicit Solvent MD Simulation Function (Optimized + GPU-aware)

# Thread lock for printing
print_lock = threading.Lock()

def thread_print(*args, **kwargs):
    """Thread-safe print."""
    with print_lock:
        print(*args, **kwargs)

def run_simulation(name, mutations, gpu_id=0):
    """
    Run MD simulation with explicit solvent (TIP3P) on specified GPU.
    OPTIMIZED with HMR (4 fs timestep) and smaller water box.
    
    Args:
        name: Variant name (e.g., 'WT', 'R175H')
        mutations: List of mutations to apply
        gpu_id: GPU device index (0, 1, etc.)
    
    Returns:
        Dictionary with simulation results
    """
    thread_print(f"\n{'='*60}")
    thread_print(f"SIMULATING: {name} (Optimized Explicit Solvent) on GPU {gpu_id}")
    thread_print(f"{'='*60}")
    
    # 1. Apply mutations to sequence
    thread_print(f"\n[{name}] Preparing sequence...")
    sequence = P53_CORE
    if mutations:
        sequence = apply_mutations(sequence, mutations)
    else:
        thread_print("  No mutations (wild-type)")
    
    # 2. Get structure from ESMFold (use cache if available)
    pdb_path = f"structures/{name}_esmfold.pdb"
    if not os.path.exists(pdb_path):
        thread_print(f"\n[{name}] Predicting structure with ESMFold...")
        pdb_string = predict_structure_esmfold(sequence)
        with open(pdb_path, 'w') as f:
            f.write(pdb_string)
    else:
        thread_print(f"\n[{name}] Using cached structure: {pdb_path}")
    
    # 3. Fix structure with PDBFixer
    thread_print(f"\n[{name}] Fixing structure with PDBFixer...")
    fixer = PDBFixer(filename=pdb_path)
    fixer.findMissingResidues()
    fixer.findNonstandardResidues()
    fixer.replaceNonstandardResidues()
    fixer.findMissingAtoms()
    fixer.addMissingAtoms()
    fixer.addMissingHydrogens(7.0)
    thread_print(f"  [{name}] Fixed structure: {fixer.topology.getNumAtoms()} atoms")
    
    # 4. Create explicit solvent system (OPTIMIZED: smaller water box)
    thread_print(f"\n[{name}] Creating explicit solvent system...")
    forcefield = ForceField('amber14-all.xml', 'amber14/tip3pfb.xml')
    
    # Add solvent with SMALLER padding for speed
    modeller = Modeller(fixer.topology, fixer.positions)
    thread_print(f"  [{name}] Adding water box ({WATER_PADDING} nm padding - optimized)...")
    modeller.addSolvent(
        forcefield,
        model='tip3p',
        padding=WATER_PADDING*nanometer,  # OPTIMIZED: 0.8 nm vs 1.0 nm
        ionicStrength=0.15*molar
    )
    thread_print(f"  [{name}] Solvated system: {modeller.topology.getNumAtoms()} atoms")
    
    # Save solvated structure
    solvated_path = f"structures/{name}_solvated.pdb"
    with open(solvated_path, 'w') as f:
        PDBFile.writeFile(modeller.topology, modeller.positions, f)
    
    # Create system
    thread_print(f"  [{name}] Creating OpenMM system...")
    system = forcefield.createSystem(
        modeller.topology,
        nonbondedMethod=PME,
        nonbondedCutoff=1.0*nanometer,
        constraints=HBonds,
        hydrogenMass=4*amu if USE_HMR else None  # HMR for 4 fs timestep
    )
    
    if USE_HMR:
        thread_print(f"  [{name}] HMR enabled (4 fs timestep)")
    
    # Setup platform with specific GPU
    if PLATFORM_NAME == 'CUDA':
        platform = Platform.getPlatformByName('CUDA')
        properties = {'DeviceIndex': str(gpu_id), 'Precision': 'mixed'}
        thread_print(f"  [{name}] Using CUDA GPU {gpu_id}")
    elif PLATFORM_NAME == 'OpenCL':
        platform = Platform.getPlatformByName('OpenCL')
        properties = {'DeviceIndex': str(gpu_id), 'Precision': 'mixed'}
        thread_print(f"  [{name}] Using OpenCL device {gpu_id}")
    else:
        platform = Platform.getPlatformByName('CPU')
        properties = {}
        thread_print(f"  [{name}] Using CPU")
    
    # 5. Energy minimization
    thread_print(f"\n[{name}] Energy minimization...")
    integrator = LangevinMiddleIntegrator(300*kelvin, 1/picosecond, TIMESTEP*femtoseconds)
    
    if properties:
        simulation = Simulation(modeller.topology, system, integrator, platform, properties)
    else:
        simulation = Simulation(modeller.topology, system, integrator, platform)
    
    simulation.context.setPositions(modeller.positions)
    
    e_initial = simulation.context.getState(getEnergy=True).getPotentialEnergy()
    simulation.minimizeEnergy(maxIterations=1000)
    e_final = simulation.context.getState(getEnergy=True).getPotentialEnergy()
    thread_print(f"  [{name}] Energy: {e_initial} -> {e_final}")
    
    positions = simulation.context.getState(getPositions=True).getPositions()
    
    # 6. NPT Equilibration
    thread_print(f"\n[{name}] NPT Equilibration ({EQUILIBRATION_STEPS * TIMESTEP / 1000:.0f} ps)...")
    
    # Add barostat for pressure control
    system.addForce(MonteCarloBarostat(1*bar, 300*kelvin))
    
    integrator = LangevinMiddleIntegrator(300*kelvin, 1/picosecond, TIMESTEP*femtoseconds)
    if properties:
        simulation = Simulation(modeller.topology, system, integrator, platform, properties)
    else:
        simulation = Simulation(modeller.topology, system, integrator, platform)
    simulation.context.setPositions(positions)
    simulation.context.setVelocitiesToTemperature(300*kelvin)
    
    # Progress reporter for equilibration
    log_file = f"trajectories/{name}_eq_log.txt"
    simulation.reporters.append(
        StateDataReporter(log_file, 5000, step=True, temperature=True, 
                          progress=True, remainingTime=True, speed=True,
                          totalSteps=EQUILIBRATION_STEPS)
    )
    
    simulation.step(EQUILIBRATION_STEPS)
    
    # Save equilibrated structure
    eq_positions = simulation.context.getState(getPositions=True).getPositions()
    eq_path = f"structures/{name}_equilibrated.pdb"
    with open(eq_path, 'w') as f:
        PDBFile.writeFile(modeller.topology, eq_positions, f)
    thread_print(f"  [{name}] Saved: {eq_path}")
    
    # 7. Production MD
    thread_print(f"\n[{name}] Production MD ({PRODUCTION_STEPS * TIMESTEP / 1e6:.1f} ns) on GPU {gpu_id}...")
    
    # Clear reporters and add new ones
    simulation.reporters.clear()
    
    traj_path = f"trajectories/{name}_traj.dcd"
    log_path = f"trajectories/{name}_log.csv"
    
    # Remove old trajectory if exists
    if os.path.exists(traj_path):
        os.remove(traj_path)
    
    simulation.reporters.append(DCDReporter(traj_path, SAVE_INTERVAL))
    simulation.reporters.append(
        StateDataReporter(log_path, SAVE_INTERVAL, step=True, time=True,
                          potentialEnergy=True, temperature=True)
    )
    # Progress to file
    simulation.reporters.append(
        StateDataReporter(f"trajectories/{name}_progress.txt", 25000, step=True, time=True,
                          potentialEnergy=True, temperature=True,
                          progress=True, remainingTime=True, speed=True,
                          totalSteps=PRODUCTION_STEPS)
    )
    
    simulation.step(PRODUCTION_STEPS)
    
    # 8. Analysis
    thread_print(f"\n[{name}] Analyzing trajectory...")
    traj = md.load(traj_path, top=eq_path)
    
    # Select protein atoms only (exclude water)
    protein_idx = traj.topology.select('protein')
    protein_traj = traj.atom_slice(protein_idx)
    thread_print(f"  [{name}] Protein atoms: {protein_traj.n_atoms}")
    
    # RMSD (protein only)
    rmsd = md.rmsd(protein_traj, protein_traj, 0) * 10  # nm to Angstrom
    
    # RMSF (CA atoms)
    protein_aligned = protein_traj.superpose(protein_traj, 0)
    ca_indices = protein_traj.topology.select('name CA')
    rmsf = md.rmsf(protein_aligned, protein_aligned, frame=0, atom_indices=ca_indices) * 10
    residue_nums = [protein_traj.topology.atom(i).residue.resSeq for i in ca_indices]
    
    # Save analysis
    np.save(f"trajectories/{name}_rmsd.npy", rmsd)
    np.save(f"trajectories/{name}_rmsf.npy", rmsf)
    
    results = {
        'name': name,
        'mutations': mutations,
        'gpu_id': gpu_id,
        'n_frames': traj.n_frames,
        'n_atoms': protein_traj.n_atoms,
        'n_atoms_total': traj.n_atoms,
        'rmsd': rmsd,
        'rmsf': rmsf,
        'residue_nums': residue_nums,
        'mean_rmsd': np.mean(rmsd),
        'final_rmsd': rmsd[-1],
        'max_rmsd': np.max(rmsd),
        'mean_rmsf': np.mean(rmsf)
    }
    
    thread_print(f"\n[{name}] Complete on GPU {gpu_id}!")
    thread_print(f"  [{name}] Frames: {traj.n_frames}")
    thread_print(f"  [{name}] Mean RMSD: {np.mean(rmsd):.2f} A")
    thread_print(f"  [{name}] Final RMSD: {rmsd[-1]:.2f} A")
    
    return results


print("Simulation function defined (OPTIMIZED: HMR + smaller water box).")

In [ ]:
# @title 6. Run All Simulations (Multi-GPU Parallel)

print("="*60)
print("RUNNING EXPLICIT SOLVENT MD SIMULATIONS (MULTI-GPU)")
print("="*60)
print(f"Method: Explicit solvent (TIP3P) + PME")
print(f"Timestep: {TIMESTEP} fs")
print(f"Equilibration: {EQUILIBRATION_STEPS * TIMESTEP / 1000:.0f} ps NPT")
print(f"Production: {PRODUCTION_STEPS * TIMESTEP / 1e6:.1f} ns per variant")
print(f"Variants: {list(VARIANTS.keys())}")
print(f"GPUs available: {N_GPUS}")
print("="*60)

# Prepare variant list with GPU assignments
variant_list = list(VARIANTS.items())
n_variants = len(variant_list)

if N_GPUS >= 2:
    # Multi-GPU: Run variants in parallel
    print(f"\n>>> PARALLEL MODE: Using {N_GPUS} GPUs simultaneously <<<")
    print(f"    GPU 0: WT, R175H_N239Y (if 3 variants)")
    print(f"    GPU 1: R175H")
    print("="*60)
    
    results = {}
    
    # First batch: Run 2 variants in parallel (one per GPU)
    with ThreadPoolExecutor(max_workers=N_GPUS) as executor:
        futures = {}
        
        # Assign variants to GPUs (round-robin)
        for i, (name, mutations) in enumerate(variant_list):
            gpu_id = i % N_GPUS
            future = executor.submit(run_simulation, name, mutations, gpu_id)
            futures[future] = name
            print(f"  Submitted {name} to GPU {gpu_id}")
        
        # Collect results as they complete
        for future in as_completed(futures):
            name = futures[future]
            try:
                results[name] = future.result()
                print(f"\n>>> {name} COMPLETED <<<")
            except Exception as e:
                print(f"\n>>> {name} FAILED: {e} <<<")
                raise

else:
    # Single GPU or CPU: Run sequentially
    print(f"\n>>> SEQUENTIAL MODE: Using single device <<<")
    print("="*60)
    
    results = {}
    for name, mutations in VARIANTS.items():
        results[name] = run_simulation(name, mutations, gpu_id=0)

print("\n" + "="*60)
print("ALL SIMULATIONS COMPLETE")
print("="*60)

# Print timing summary
for name, r in results.items():
    print(f"  {name}: GPU {r.get('gpu_id', 0)}, Final RMSD: {r['final_rmsd']:.2f} A")

In [ ]:
# @title 7. Generate Comparison Plots

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Colors for each variant
colors = {
    'WT': 'green', 
    'R175H': 'red', 
    'R175H_S95A': 'blue',
    'R175H_M133L': 'purple',
    'R175H_C229A': 'orange',
}
labels = {
    'WT': 'Wild-type',
    'R175H': 'R175H (cancer)',
    'R175H_S95A': '+S95A (ΔΔG: -6.23)',
    'R175H_M133L': '+M133L (ΔΔG: -5.60)',
    'R175H_C229A': '+C229A (ΔΔG: -4.40)',
}

# Plot 1: RMSD over time
ax1 = axes[0, 0]
for name, r in results.items():
    time_ns = np.arange(len(r['rmsd'])) * TIMESTEP * SAVE_INTERVAL / 1e6
    ax1.plot(time_ns, r['rmsd'], color=colors.get(name, 'gray'), linewidth=1.5, 
             label=labels.get(name, name), alpha=0.8)

ax1.axhline(y=2.5, color='black', linestyle='--', linewidth=2, label='Stability threshold')
ax1.set_xlabel('Time (ns)', fontsize=12)
ax1.set_ylabel('RMSD (Å)', fontsize=12)
ax1.set_title('Protein Stability Over Time', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left', fontsize=9)
ax1.grid(True, alpha=0.3)

# Plot 2: Final RMSD bar chart
ax2 = axes[0, 1]
names = list(results.keys())
final_rmsds = [results[n]['final_rmsd'] for n in names]
bar_colors = [colors.get(n, 'gray') for n in names]

bars = ax2.bar(range(len(names)), final_rmsds, color=bar_colors, alpha=0.7, edgecolor='black', linewidth=2)
ax2.axhline(y=2.5, color='black', linestyle='--', linewidth=2)
ax2.set_xticks(range(len(names)))
ax2.set_xticklabels([labels.get(n, n) for n in names], rotation=45, ha='right', fontsize=9)

for bar, val in zip(bars, final_rmsds):
    color = 'green' if val < 2.5 else 'red'
    ax2.text(bar.get_x() + bar.get_width()/2., val + 0.1,
             f'{val:.2f}', ha='center', fontsize=10, fontweight='bold', color=color)

ax2.set_ylabel('Final RMSD (Å)', fontsize=12)
ax2.set_title('Final Stability Comparison', fontsize=14, fontweight='bold')
ax2.set_ylim(0, max(final_rmsds) * 1.3)

# Plot 3: RMSF comparison
ax3 = axes[1, 0]
for name, r in results.items():
    ax3.plot(r['residue_nums'], r['rmsf'], color=colors.get(name, 'gray'),
             linewidth=1, label=labels.get(name, name), alpha=0.7)

ax3.axvline(x=175, color='red', linestyle=':', linewidth=2, alpha=0.7)
ax3.set_xlabel('Residue Number', fontsize=12)
ax3.set_ylabel('RMSF (Å)', fontsize=12)
ax3.set_title('Per-Residue Flexibility', fontsize=14, fontweight='bold')
ax3.legend(loc='upper right', fontsize=8)
ax3.grid(True, alpha=0.3)

# Plot 4: Summary table
ax4 = axes[1, 1]
ax4.axis('off')

table_data = [['Variant', 'Predicted ΔΔG', 'Final RMSD', 'Stable?']]
ddg_predictions = {'WT': 'baseline', 'R175H': '+9.99', 
                   'R175H_S95A': '-6.23', 'R175H_M133L': '-5.60', 'R175H_C229A': '-4.40'}

for name in names:
    r = results[name]
    stable = '✓ YES' if r['final_rmsd'] < 2.5 else '✗ NO'
    table_data.append([
        labels.get(name, name),
        ddg_predictions.get(name, 'N/A'),
        f"{r['final_rmsd']:.2f} Å",
        stable
    ])

table = ax4.table(cellText=table_data, loc='center', cellLoc='center',
                  colWidths=[0.35, 0.2, 0.2, 0.15])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 2.2)

for j in range(4):
    table[(0, j)].set_facecolor('#2E86AB')
    table[(0, j)].set_text_props(color='white', fontweight='bold')

for i, name in enumerate(names, 1):
    if results[name]['final_rmsd'] < 2.5:
        table[(i, 3)].set_facecolor('#C6EFCE')
    else:
        table[(i, 3)].set_facecolor('#FFC7CE')

ax4.set_title('p53CAD Rescue Validation Summary', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.savefig('trajectories/p53cad_validation_10ns.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nPlot saved: trajectories/p53cad_validation_10ns.png")

In [ ]:
# @title 8. Final Results & Conclusions

print("="*70)
print("p53CAD: MD VALIDATION RESULTS")
print("="*70)

print(f"\nSimulation: {PRODUCTION_STEPS * TIMESTEP / 1e6:.0f} ns explicit solvent (TIP3P)")
print(f"Method: AMBER14 forcefield, PME electrostatics, HMR (4 fs timestep)")

print(f"\n{'='*70}")
print("STABILITY RANKING (by Final RMSD)")
print("="*70)

sorted_variants = sorted(results.items(), key=lambda x: x[1]['final_rmsd'])
ddg_pred = {'WT': 'baseline', 'R175H': '+9.99', 
            'R175H_S95A': '-6.23', 'R175H_M133L': '-5.60', 'R175H_C229A': '-4.40'}

for rank, (name, r) in enumerate(sorted_variants, 1):
    stable = "STABLE" if r['final_rmsd'] < 2.5 else "UNSTABLE"
    icon = "✓" if r['final_rmsd'] < 2.5 else "✗"
    print(f"\n#{rank} {labels.get(name, name)}")
    print(f"    Predicted ΔΔG: {ddg_pred.get(name, 'N/A')} kcal/mol")
    print(f"    Final RMSD:    {r['final_rmsd']:.2f} Å  {icon} {stable}")
    print(f"    Mean RMSD:     {r['mean_rmsd']:.2f} Å")

print(f"\n{'='*70}")
print("CONCLUSIONS")
print("="*70)

wt_rmsd = results['WT']['final_rmsd']
r175h_rmsd = results['R175H']['final_rmsd']

# Count successful rescues
successful = sum(1 for n, r in results.items() if n.startswith('R175H_') and r['final_rmsd'] < 2.5)
total_rescues = sum(1 for n in results if n.startswith('R175H_'))

print(f"\n1. CANCER MUTATION EFFECT:")
print(f"   R175H destabilizes p53: {wt_rmsd:.2f} → {r175h_rmsd:.2f} Å (+{r175h_rmsd-wt_rmsd:.2f} Å)")

print(f"\n2. RESCUE SUCCESS RATE: {successful}/{total_rescues}")

# Correlation analysis
print(f"\n3. PREDICTION vs SIMULATION CORRELATION:")
for name, r in results.items():
    if name.startswith('R175H_'):
        pred = float(ddg_pred.get(name, '0').replace('+', ''))
        improvement = r175h_rmsd - r['final_rmsd']
        print(f"   {name.replace('R175H_', '')}: Predicted {pred:+.2f} → MD improvement {improvement:+.2f} Å")

best = min([(n, r) for n, r in results.items() if n.startswith('R175H_')], 
           key=lambda x: x[1]['final_rmsd'])
print(f"\n>>> BEST RESCUE: {best[0].replace('R175H_', '')} (Final RMSD: {best[1]['final_rmsd']:.2f} Å) <<<")

print(f"\n{'='*70}")
print("FILES GENERATED")
print("="*70)
print("structures/     - PDB files (ESMFold predicted, solvated, equilibrated)")
print("trajectories/   - DCD trajectories, RMSD/RMSF data, analysis plots")
print("\nDownload: !zip -r results.zip structures/ trajectories/")

## Notes on Explicit Solvent Simulations

### Advantages over Implicit Solvent
- **More accurate** - Explicit water molecules capture hydrogen bonding and hydration effects
- **Better for stability** - More reliable RMSD measurements
- **Industry standard** - Used in all publication-quality MD studies

### Computational Cost
- System size: ~100,000 atoms (vs ~3,500 for implicit)
- Speed: ~30-50x slower than implicit solvent
- Recommended: Use GPU acceleration (Colab T4 or better)

### For Best Results
1. Run 10-50 ns production (increase PRODUCTION_STEPS)
2. Run 3-5 independent replicates (different random seeds)
3. Analyze last 50% of trajectory (after equilibration)
4. Compare with experimental melting temperatures if available

### References
- AMBER14 forcefield: Maier et al., JCTC 2015
- TIP3P water model: Jorgensen et al., JCP 1983
- p53 stability: Bullock et al., PNAS 2000